# Recorrido completo del TPI — paso a paso

Este notebook recorre **todo** el pipeline sobre un nódulo real, mostrando cada paso con imágenes y explicando el porqué.
Usa los mismos módulos de `tpi/` que los scripts, así que lo que ves acá es exactamente lo que hacen los scripts 01–05.

**Cómo usarlo:** ejecutá las celdas en orden (`Shift + Enter`). Para mirar otro nódulo, cambiá `CASO` en la celda de configuración y volvé a ejecutar desde ahí.

## El mapa general

```
 PREGUNTA: ¿qué medidas del nódulo (adentro y alrededor) distinguen sospechosos de benignos, más allá del tamaño?

 1. Selección ............ de 2651 nódulos de LIDC nos quedamos con sólidos, ≥6 mm, ≥3 radiólogos, malignidad clara
 2. Descarga ............. solo las tomografías elegidas
 3. Corte de trabajo ..... el corte donde el nódulo se ve más grande, remuestreado a 0,7 mm, recorte de 64 mm
 4. Pre-procesamiento .... (obligatorio por consigna) reconstrucción simulada, filtros, compresión
 5. Segmentación ......... desde un punto (semilla) dentro del nódulo
 6. Validación ........... ¿se parece al contorno de los radiólogos tanto como ellos entre sí?
 7. Características ...... tamaño, forma, intensidad, composición, textura + anillos de tejido alrededor
 8. Estadística .......... ¿qué medidas difieren entre grupos? (cuando haya más nódulos)
```

In [ ]:
import sys, glob
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage import measure, morphology
from skimage.feature import graycomatrix

from tpi import config as C
from tpi import segmentacion as S, validacion as V, caracteristicas as F
from tpi import entorno as E, reconstruccion as R, compresion as Z, mejoramiento as M
from tpi.datos import cargar_corte, _recortar

plt.rcParams.update({"figure.dpi": 110, "image.cmap": "gray", "axes.titlesize": 9})
VENT = dict(vmin=-1000, vmax=400, cmap="gray")   # ventana de pulmón

def contorno(ax, m, color, lw=1.5, ls="-"):
    if m is not None and m.any():
        ax.contour(m.astype(float), levels=[0.5], colors=[color], linewidths=lw, linestyles=ls)

def sin_ejes(axs):
    for a in np.ravel(axs): a.axis("off")

## Configuración: qué nódulo mirar

Si ya corriste el script 03, usa tus cortes reales (`datos/cortes/`). Si no, usa el fantoma sintético (`tests/salida/cortes/`).

El umbral de segmentación se toma del ajuste del script 04 (`resultados/04_ajuste_umbral.csv`); si no existe, del `config.py`.

In [ ]:
archivos = sorted((RAIZ / "datos" / "cortes").glob("*.npz"))
FUENTE = "LIDC real"
if not archivos:
    archivos = sorted((RAIZ / "tests" / "salida" / "cortes").glob("*.npz"))
    FUENTE = "fantoma sintético"
print(f"{len(archivos)} cortes ({FUENTE})")

ajuste = RAIZ / "resultados" / "04_ajuste_umbral.csv"
_aj = pd.read_csv(ajuste).sort_values("dice_medio").iloc[-1] if ajuste.exists() else None
T = int(_aj["umbral_hu"]) if _aj is not None else C.UMBRAL_HU
RC = float(_aj["radio_cierre_mm"]) if _aj is not None and "radio_cierre_mm" in _aj else C.RADIO_CIERRE_PULMON_MM
print("Umbral de segmentación T =", T, "HU | cierre del pulmón:", RC, "mm")

sel_csv = RAIZ / "resultados" / "nodulos_seleccionados.csv"
meta = pd.read_csv(sel_csv) if sel_csv.exists() else pd.DataFrame()

def info(ruta):
    import re
    m = re.match(r"(?P<p>.+)_s(?P<s>\d+)_n(?P<n>\d+)$", ruta.stem)
    fila = meta[(meta.get("scan_id") == int(m["s"])) & (meta.get("nodulo") == int(m["n"]))] if len(meta) else []
    return fila.iloc[0].to_dict() if len(fila) else {}

for i, a in enumerate(archivos):
    f = info(a)
    print(f"{i:2d}  {a.stem:28s} {str(f.get('grupo', '?')):11s} malig={f.get('malignidad_mediana', '?')}  diám={f.get('diametro_medio_mm', float('nan')):.1f} mm")

In [ ]:
CASO = 0          # <-- cambiá este número para ver otro nódulo
d = cargar_corte(archivos[CASO])
inf = info(archivos[CASO])
img, sem = d["img"], d["semilla"]
print(archivos[CASO].stem, "| grupo:", inf.get("grupo"), "| píxel original:", float(d["pixel_mm_original"]), "mm")

---
# 1. El dataset y la selección

**LIDC-IDRI**: ~1000 tomografías de tórax. Cada una la leyeron **4 radiólogos reales**, cada uno por su cuenta: marcaron los nódulos ≥ 3 mm, **dibujaron el contorno en cada corte** y les pusieron notas de 1 a 5 (malignidad, textura, espiculación, calcificación…).

`pylidc` trae todas esas anotaciones en una base de datos local, por eso pudimos **elegir los nódulos sin bajar imágenes**. Cada filtro tiene un porqué:

| Filtro | Por qué |
|---|---|
| Espesor de corte ≤ 2,5 mm | cortes finos → el corte de mayor área representa bien al nódulo |
| ≤ 4 anotaciones por grupo | más de 4 = dos nódulos pegados mezclados (no sabemos qué marca es de cuál) |
| ≥ 3 radiólogos | con menos no hay consenso ni concordancia confiable |
| Sólido (textura mediana ≥ 4) | los subsólidos (vidrio esmerilado) se segmentan muy distinto |
| Diámetro ≥ 6 mm | por debajo, Lung-RADS no cambia la conducta clínica |
| Malignidad ≤ 2 o ≥ 4 | dejamos afuera la "zona gris" (3): grupos claros |

In [ ]:
flujo = RAIZ / "resultados" / "flujo_seleccion.csv"
if flujo.exists():
    f = pd.read_csv(flujo)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.barh(f["paso"][::-1], f["quedan"][::-1], color="steelblue")
    for y, v in enumerate(f["quedan"][::-1]): ax.text(v + 20, y, str(v), va="center", fontsize=8)
    ax.set_xlabel("nódulos que quedan"); ax.set_title("Flujo de selección")
    plt.show()
if len(meta):
    print(pd.crosstab(meta["grupo"], meta["conjunto"], margins=True))

**División entrenamiento / evaluación (80/20) por paciente.** Si dos nódulos del mismo paciente cayeran uno en cada conjunto, el de evaluación ya no sería "nuevo" (mismo tomógrafo, mismo pulmón). Todo lo que se ajusta (por ejemplo, el umbral T) se ajusta **solo** con entrenamiento; evaluación se usa una vez, al final.

---
# 2. El corte de trabajo

**Unidades Hounsfield (HU):** la tomografía mide cuánto atenúa los rayos X cada punto, en una escala donde **aire = −1000**, **agua = 0**. Valores típicos: pulmón ≈ −850, grasa ≈ −100, tejido blando/nódulo ≈ 0 a 50, calcio/hueso > 200.

Para cada nódulo:
1. Se arma el **consenso**: un píxel es del nódulo si lo marcó ≥ 50 % de los radiólogos.
2. Se elige el **corte axial donde el consenso tiene mayor área**.
3. Se **remuestrea** a 0,7 mm por píxel (cada tomógrafo tiene su tamaño de píxel; si no, "10 píxeles" significaría cosas distintas).
4. **Semilla**: el píxel del consenso más cercano a su centro (simula al médico haciendo clic sobre el nódulo).
5. **Recorte de 64 × 64 mm** alrededor de la semilla.

In [ ]:
lado = img.shape[0]
sc = d["semilla_completo"]
fig, axs = plt.subplots(1, 3, figsize=(13, 4.5))
axs[0].imshow(d["corte_completo_rem"], **VENT)
axs[0].add_patch(plt.Rectangle((sc[1] - lado/2, sc[0] - lado/2), lado, lado, fill=False, ec="red", lw=1.5))
axs[0].set_title("Corte completo (ventana de pulmón) + recorte de 64 mm")
axs[1].imshow(img, **VENT); axs[1].plot(sem[1], sem[0], "c+", ms=14, mew=2)
axs[1].set_title("Recorte 64 mm + semilla (+)")
axs[2].imshow(img, vmin=-1400, vmax=1400, cmap="gray")
axs[2].set_title("Mismo recorte, ventana ancha [-1400, 1400]")
sin_ejes(axs); plt.show()

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.hist(img.ravel(), bins=150, range=(-1100, 500), color="gray")
for v, t in [(-850, "pulmón"), (0, "agua/tejido"), (T, f"umbral T={T}")]:
    ax.axvline(v, color="r" if t.startswith("umbral") else "k", ls="--", lw=1); ax.text(v, ax.get_ylim()[1]*0.85, t, rotation=90, fontsize=8)
ax.set_xlabel("HU"); ax.set_title("Histograma del recorte: dos poblaciones, aire del pulmón y tejido"); plt.show()

**¿Por qué "ventana"?** La pantalla muestra 256 grises, pero las HU van de −1000 a más de 3000. La ventana elige qué rango se reparte entre esos grises (es un estiramiento de contraste, como la ecualización pero lineal y fijo). Con la ventana de pulmón [−1000, 400] se ven bien el parénquima y el nódulo; con la ancha todo queda lavado. **No usamos ecualización**: cambia los valores de cada imagen de forma distinta, y nosotros necesitamos que las HU signifiquen lo mismo en todos los nódulos.

---
# 2bis. Análisis: mirar antes de tocar

Paso 1 del pipeline de la materia. Antes de elegir métodos, miramos **todas** las regiones:

- **Forma del histograma** → si es **bimodal** (aire del pulmón vs tejido), un umbral separa bien las dos clases → justifica umbral / Otsu / crecimiento de regiones por umbral.
- **Tipo de ruido** → granulado parejo (gaussiano) o puntos sueltos (impulsivo). Eso decide el filtro (gaussiano vs mediana).
- **¿Hace falta filtrar antes de segmentar?** Si el ruido es chico comparado con el salto pulmón–tejido (~850 HU), no. Y como después medimos textura, la materia dice: *si no hace falta filtrar, no filtres*.

In [ ]:
from skimage.filters import threshold_otsu
def ruido_parenquima(im):
    res = im - ndi.gaussian_filter(im, 2); m = (im > -950) & (im < -700)
    r = res[m]; return 1.4826 * np.median(np.abs(r - np.median(r)))
bins = np.linspace(-1100, 500, 81); otsus, ruidos = [], []
fig, axs = plt.subplots(1, 3, figsize=(16, 3.6))
for a_ in archivos:
    dd = cargar_corte(a_); h, _ = np.histogram(dd["img"], bins=bins, density=True)
    axs[0].plot((bins[:-1] + bins[1:]) / 2, h, alpha=.5, lw=1)
    otsus.append(threshold_otsu(np.clip(dd["img"], -1000, 400))); ruidos.append(ruido_parenquima(dd["img"]))
axs[0].set_yscale("log"); axs[0].set_xlabel("HU"); axs[0].set_title("Histograma de cada región: bimodal (aire / tejido)")
axs[1].hist(otsus, bins=10, color="gray"); axs[1].axvline(T, color="r", ls="--", label=f"nuestro T={T}")
axs[1].set_title("Umbral de Otsu de cada región"); axs[1].set_xlabel("HU"); axs[1].legend(fontsize=8)
res = img - ndi.gaussian_filter(img, 2)
axs[2].imshow(res, cmap="RdBu", vmin=-80, vmax=80); axs[2].axis("off")
axs[2].set_title(f"Residuo (imagen - suavizada) de este caso: granulado\nruido ≈ {ruido_parenquima(img):.0f} HU")
plt.tight_layout(); plt.show()
print(f"Ruido mediano en parénquima: {np.median(ruidos):.0f} HU, contra un salto pulmón-tejido de ~850 HU "
      f"-> no hace falta filtrar antes de segmentar.")

**Conclusión del análisis:** histogramas bimodales → segmentar por umbral tiene sentido; Otsu da un umbral parecido en todas las regiones (pero no sabe cuál mancha es el nódulo → hace falta la semilla); el ruido es granulado y chico frente al contraste → no filtramos antes de segmentar ni de medir textura.

---
# 3. Los radiólogos no coinciden entre sí

Esta es la idea central del proyecto. Abajo: el contorno de cada radiólogo en este corte, y el **Dice** entre cada par.

**Dice** = 2·|A∩B| / (|A|+|B|). Vale 1 si dos contornos son idénticos y 0 si no se tocan.

La **concordancia entre radiólogos** de este nódulo es el promedio de esos Dice de a pares. Ese número es la vara: le pedimos a nuestro algoritmo que se parezca al consenso **al menos tanto como los radiólogos se parecen entre sí**.

In [ ]:
rads = [m for m in d["radiologos"] if m.any()]
colores = ["yellow", "cyan", "magenta", "orange"]
n = len(rads)
fig, axs = plt.subplots(1, n + 2, figsize=(3.2 * (n + 2), 3.4))
zoom = lambda a: a  # el recorte ya está centrado
for i, m in enumerate(rads):
    axs[i].imshow(img, **VENT); contorno(axs[i], m, colores[i]); axs[i].set_title(f"Radiólogo {i+1}")
axs[n].imshow(img, **VENT)
for i, m in enumerate(rads): contorno(axs[n], m, colores[i], 1)
contorno(axs[n], d["consenso"], "lime", 2.2); axs[n].set_title("Todos + consenso (verde)")
D = np.array([[V.dice(a, b) for b in rads] for a in rads])
im = axs[n+1].imshow(D, vmin=0.5, vmax=1, cmap="viridis")
for i in range(n):
    for j in range(n): axs[n+1].text(j, i, f"{D[i,j]:.2f}", ha="center", va="center", color="w", fontsize=8)
axs[n+1].set_xticks(range(n)); axs[n+1].set_yticks(range(n))
axs[n+1].set_xticklabels([f"R{i+1}" for i in range(n)]); axs[n+1].set_yticklabels([f"R{i+1}" for i in range(n)])
axs[n+1].set_title("Dice entre radiólogos")
sin_ejes(axs[:n+1]); plt.tight_layout(); plt.show()
print(f"Concordancia entre radiólogos = {V.concordancia_radiologos(d['radiologos']):.3f}")

---
# 4. Segmentación, paso a paso

**Por qué crecimiento de regiones** (tabla de decisión de la materia): es **un** objeto, bastante homogéneo, y **sabemos dónde está** (la semilla). Otsu solo no alcanza: separa "tejido" de "aire", pero no sabe cuál de las manchas de tejido es el nódulo (vasos, pared y otros nódulos también son tejido).

**Criterio de semejanza.** En la materia es "diferencia con la semilla ≤ tolerancia". Usamos **"HU > T"** (pertenecer al rango de tejido): no depende del valor de un único píxel (la semilla podría caer sobre un punto de calcio o de ruido). Con un criterio fijo, crecer desde la semilla agregando vecinos (conectividad 8) da exactamente *"la mancha conexa de (imagen > T) que contiene a la semilla"*, que se obtiene con **etiquetado**.

Los pasos:
0. **Pulmón cerrado** ("de lo grande a lo chico", materia 4.10): primero se segmenta el pulmón (aire que no toca el borde de la imagen) y se le aplica un **cierre con un disco grande** (~30 mm). Un nódulo pegado a la pared forma una muesca en el borde del pulmón; el cierre la rellena sin meterse en la pared. El nódulo solo puede crecer adentro de esta región. Sin esto, con datos reales ~30 % de los nódulos (los pegados a la pared) se "escapaban" por toda la pared del tórax.
1. **Umbral**: todo lo que es más denso que T (nódulo, pero también vasos, pared, etc.), dentro del pulmón cerrado.
2. **Componente con la semilla**: de todas esas manchas, la que toca a la semilla.
3. **Radio máximo (30 mm)**: freno extra para fugas que queden.
4. **Apertura morfológica** (erosión + dilatación con un disco chico): corta conexiones finas, como un vaso pegado al nódulo.
5. **Relleno de huecos**: si adentro hay aire o zonas de menor densidad, se incluyen (son parte del nódulo y queremos medirlas). No usamos **cierre** porque además suavizaría el borde y taparía los entrantes entre lóbulos, justo lo que mide la forma.

In [ ]:
permitida = S.region_permitida(d, RC)
paso0 = permitida if permitida is not None else np.ones_like(img, bool)
paso1 = (img > T) & paso0
lab = measure.label(paso1, connectivity=2)
paso2 = lab == lab[sem]
rr, cc = np.ogrid[:lado, :lado]
dentro = (rr - sem[0])**2 + (cc - sem[1])**2 <= (C.RADIO_MAX_MM / C.PIXEL_MM)**2
paso3 = S._componente_con_semilla(paso1 & dentro, sem)
paso4 = S._componente_con_semilla(ndi.binary_opening(paso3, structure=morphology.disk(C.RADIO_APERTURA_PX)), sem)
paso5 = ndi.binary_fill_holes(paso4)
seg = S.segmentar_nodulo(d, umbral=T, radio_cierre_mm=RC)
assert np.array_equal(seg, paso5)

fig, axs = plt.subplots(1, 7, figsize=(19, 3.2))
axs[0].imshow(img, **VENT); axs[0].set_title("Original")
for a, m, t in zip(axs[1:], [paso0, paso1, paso2, paso3, paso4, paso5],
                   [f"0. Pulmón cerrado ({RC:g} mm)", f"1. Umbral > {T} HU", "2. Mancha con la semilla",
                    "3. + radio máx 30 mm", "4. + apertura (corta vasos)", "5. + relleno de huecos"]):
    a.imshow(img, **VENT); a.imshow(np.ma.masked_where(~m, m), cmap="autumn", alpha=0.5); a.set_title(t)
    a.plot(sem[1], sem[0], "c+", ms=8)
sin_ejes(axs); plt.tight_layout(); plt.show()

**¿Cómo se elige T?** El borde del nódulo no es nítido: hay una transición borrosa entre pulmón (−850) y nódulo (~0) por el *efecto de volumen parcial* (un píxel del borde tiene mitad aire y mitad tejido). El umbral define en qué punto de esa transición cortamos. Lo elegimos probando valores en el conjunto de entrenamiento y quedándonos con el que más se parece a los radiólogos. Abajo, cómo cambia la segmentación de **este** nódulo con distintos T:

In [ ]:
Ts = [-700, -600, -500, -400, -300, -200]
fig, axs = plt.subplots(1, len(Ts), figsize=(16, 3))
for a, t in zip(axs, Ts):
    s_ = S.segmentar_nodulo(d, umbral=t, radio_cierre_mm=RC)
    a.imshow(img, **VENT); contorno(a, d["consenso"], "lime"); contorno(a, s_, "red")
    a.set_title(f"T={t}  Dice={V.dice(s_, d['consenso']):.2f}")
sin_ejes(axs); plt.tight_layout(); plt.show()

if ajuste.exists():
    aj = pd.read_csv(ajuste)
    fig, ax = plt.subplots(figsize=(6, 2.8))
    for r_, sub in aj.groupby(aj.get("radio_cierre_mm", pd.Series(RC, index=aj.index))):
        ax.plot(sub.umbral_hu, sub.dice_medio, "o-", label=f"Dice medio, cierre {r_:g} mm")
    ax.axvline(T, color="r", ls=":"); ax.set_xlabel("umbral T (HU)"); ax.legend(fontsize=8)
    ax.set_title("Ajuste del umbral en los nódulos de entrenamiento"); plt.show()

---
# 5. Validación: ¿la segmentación es aceptable?

Comparamos nuestra máscara (rojo) con el consenso (verde) y miramos dónde coinciden y dónde no. **Aceptable** = Dice(algoritmo, consenso) ≥ concordancia entre radiólogos de ese nódulo.

También comparamos contra **Otsu**, un método clásico que elige el umbral solo mirando el histograma (sin saber dónde está el nódulo ni cómo marcan los radiólogos).

In [ ]:
val = V.evaluar(seg, d["consenso"], d["radiologos"])
otsu, t_otsu = S.segmentar_otsu(img, sem)
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
axs[0].imshow(img, **VENT); contorno(axs[0], d["consenso"], "lime", 2); contorno(axs[0], seg, "red", 2)
axs[0].set_title(f"Algoritmo (rojo) vs consenso (verde)\nDice {val['dice']:.3f}  |  radiólogos {val['concordancia_radiologos']:.3f}")
mapa = np.zeros(img.shape + (3,))
mapa[seg & d["consenso"]] = (0, .8, 0); mapa[seg & ~d["consenso"]] = (1, 0, 0); mapa[~seg & d["consenso"]] = (0, .4, 1)
axs[1].imshow(img, **VENT); axs[1].imshow(mapa, alpha=0.6)
axs[1].set_title("verde = coinciden | rojo = sobra | azul = falta")
axs[2].imshow(img, **VENT); contorno(axs[2], d["consenso"], "lime", 2); contorno(axs[2], otsu, "orange", 2)
axs[2].set_title(f"Otsu (T={t_otsu:.0f} HU): Dice {V.dice(otsu, d['consenso']):.3f}")
sin_ejes(axs); plt.tight_layout(); plt.show()
print("ACEPTABLE" if val["aceptable"] else "NO ACEPTABLE", "| Jaccard:", round(val["jaccard"], 3))

---
# 6. Características del interior

Todo se mide **en mm y HU**, no en píxeles, para que sea comparable entre tomógrafos.

| Grupo | Qué mide | Intuición clínica |
|---|---|---|
| **Tamaño** (materia) | área, diámetro equivalente (el de un círculo de igual área) | lo que usa Lung-RADS; además es el confundidor a controlar |
| **Forma** (construido encima) | circularidad = 4π·área/perímetro², solidez = área / envolvente convexa (las dos sobre un **contorno sub-píxel**: contar píxeles de borde subestima el perímetro de los nódulos chicos y daba circularidad > 1), excentricidad (alargamiento) | bordes lobulados o espiculados → menos circular y menos sólido |
| **Espículas** (construido sobre la apertura) | `frac_fina` = fracción del área que la apertura con disco de 2 px **borra** | lo que tiene menos de ~3 mm de ancho: espículas |
| **Borde** (construido sobre Sobel) | `nitidez_borde_hu_mm` = gradiente medio sobre el contorno | borde nítido vs difuso (el "margen" de los radiólogos) |
| **Intensidad** (materia) | media y desvío de la región (máscara × original) | densidad y heterogeneidad interna |
| **Forma del histograma** (agregado) | asimetría y curtosis de las HU | colas por necrosis, calcio o burbujas, que la media y el desvío no ven (con pocos píxeles pueden ser ruidosas) |
| **Composición** (agregado, TC) | fracción > 200 HU (calcio), fracción < −600 HU (aire) | calcio → suele ser benigno; aire/burbujas → puede ser sospechoso |
| **Textura** (materia) | GLCM: energía, entropía, contraste, homogeneidad, correlación, cluster shade y prominence | cómo cambia la intensidad de un píxel a su vecino |

La forma se agrega porque la materia solo ve área y bounding box, y el signo radiológico clásico de malignidad es el **borde** (espiculado, lobulado). Circularidad y solidez son las dos medidas más simples que lo capturan.

In [ ]:
seg, cruda = S.segmentar_nodulo(d, umbral=T, devolver_cruda=True, radio_cierre_mm=RC)
car = F.caracteristicas_interior(img, seg, mascara_cruda=cruda)
props = measure.regionprops(seg.astype(np.uint8))[0]
hull = props.image_convex
r0, c0, r1, c1 = props.bbox
hull_full = np.zeros_like(seg); hull_full[r0:r1, c0:c1] = hull

fig, axs = plt.subplots(1, 4, figsize=(16, 3.8))
axs[0].imshow(img, **VENT); contorno(axs[0], seg, "red", 2); contorno(axs[0], hull_full, "yellow", 1.5, "--")
axs[0].set_title(f"Forma: nódulo (rojo) y envolvente convexa (amarillo)\nsolidez {car['solidez']:.2f} | circularidad {car['circularidad']:.2f}")
axs[1].hist(img[seg], bins=40, color="tomato"); axs[1].axvline(C.HU_CALCIO, color="k", ls="--"); axs[1].axvline(C.HU_AIRE, color="b", ls="--")
axs[1].set_title(f"Intensidad dentro del nódulo\nmedia {car['hu_media']:.0f} HU, desvío {car['hu_desvio']:.0f}"); axs[1].set_xlabel("HU")
comp = np.zeros(img.shape + (3,)); comp[seg & (img > C.HU_CALCIO)] = (1, 1, 0); comp[seg & (img < C.HU_AIRE)] = (0, .6, 1)
axs[2].imshow(img, **VENT); axs[2].imshow(comp, alpha=0.8); contorno(axs[2], seg, "red", 1)
axs[2].set_title(f"Composición: calcio (amarillo) {100*car['frac_calcio']:.1f} %\naire (celeste) {100*car['frac_aire']:.1f} %")
q = F.cuantizar(img); axs[3].imshow(np.where(seg, q, np.nan), cmap="viridis"); contorno(axs[3], seg, "red", 1)
axs[3].set_title(f"Cuantizada a {C.NIVELES_GLCM} niveles en {C.VENTANA_TEXTURA_NODULO} HU")
sin_ejes([axs[0], axs[2], axs[3]]); plt.tight_layout(); plt.show()

fig, axs = plt.subplots(1, 2, figsize=(8, 3.8))
for a, m, t in ((axs[0], cruda, "sin apertura"), (axs[1], seg, "con apertura (final)")):
    f = F.forma_y_tamano(m)
    a.imshow(img, **VENT); contorno(a, m, "red", 1.5)
    a.set_title(f"{t}\ncircularidad {f['circularidad']:.2f} | solidez {f['solidez']:.2f}")
sin_ejes(axs); plt.tight_layout(); plt.show()

**Apertura vs espículas.** La materia avisa que la apertura **borra estructuras delgadas**. La usamos para cortar vasos pegados, pero las **espículas** también son finas, y son un signo de malignidad. Por eso la forma se mide dos veces: con la máscara final y con la máscara **sin apertura** (`circularidad_cruda`, `solidez_cruda`). Si la versión cruda separa mejor los grupos, la apertura estaba borrando información.

### Cómo funciona la textura (GLCM)

La **matriz de coocurrencia** cuenta, para cada par de niveles de gris (i, j), cuántas veces un píxel de nivel *i* tiene a distancia *d* un vecino de nivel *j*. Siguiendo a la materia: **distancia 1** (textura fina), **4 direcciones** (0°, 45°, 90°, 135°) **promediadas** (el tejido no tiene una orientación preferida), **simetrizada** y **normalizada**.

**Niveles de gris: 16** (la materia dice 8 a 16; con más, la matriz queda rala — un nódulo de 6 mm tiene ~60 píxeles para llenar la matriz). **Decisión propia:** el rango que se reparte en esos 16 niveles es **[−500, 300] HU** para el nódulo y **[−1000, 0] HU** para el entorno, igual para todos los nódulos. Con la ventana completa [−1000, 400], cada nivel valdría 87 HU y todo el interior de un nódulo sólido caería en 2 o 3 niveles: la textura daría "plana" para todos.

- Si el nódulo es **homogéneo**, casi todos los pares son (i, i): la matriz se concentra en la **diagonal** → homogeneidad alta, contraste bajo.
- Si es **heterogéneo**, aparecen pares (i, j) lejanos: la matriz se **desparrama** → contraste y entropía altos.

Solo contamos pares con los dos píxeles **dentro** del nódulo: a los de afuera les damos un nivel extra (el 17º) y después tiramos esa fila y columna.

In [ ]:
qm = np.where(seg, F.cuantizar(img), C.NIVELES_GLCM).astype(np.uint8)
P = graycomatrix(qm, list(C.DISTANCIAS_GLCM), list(C.ANGULOS_GLCM), levels=C.NIVELES_GLCM + 1, symmetric=True)
P = P[:C.NIVELES_GLCM, :C.NIVELES_GLCM].sum(axis=(2, 3)).astype(float); P /= P.sum()
fig, ax = plt.subplots(figsize=(4.3, 4))
ax.imshow(np.log10(P + 1e-6), cmap="magma"); ax.set_xlabel("nivel j"); ax.set_ylabel("nivel i")
ax.set_title("GLCM del nódulo (log)"); plt.show()
print({k: round(v, 3) for k, v in car.items() if k.startswith("glcm")})

---
# 7. Entorno: el tejido alrededor del nódulo

Hipótesis 2: el parénquima que rodea al nódulo también aporta información (espículas, vasos que convergen, retracción).

1. **Máscara de pulmón**: aire (< −400 HU) que no toca el borde de la imagen (el aire de afuera del cuerpo sí lo toca), sin manchas chicas, con cierre y relleno para incluir vasos y nódulos.
2. **Anillos por dilatación** (la materia lo menciona: "margen alrededor de un tumor"): interno = dilatar el nódulo con un disco de 3 mm y restar el nódulo; externo = dilatar con 6 mm y restar la dilatación de 3 mm. **Solo dentro del pulmón**. Distancias fijas en mm → comparamos la misma franja de tejido en nódulos chicos y grandes.

   Se calcula con la **transformada de distancia** (cada píxel sabe a cuántos mm está del nódulo): dilatar con un disco de radio r es exactamente quedarse con los puntos a distancia ≤ r. Con un disco hecho de píxeles de 0,7 mm, 3 mm se redondearía a 2,8 y 6 mm a 6,3, y los anillos cambiarían un 11–17 %.

In [ ]:
pulmon_full = E.mascara_pulmon(d["corte_completo_rem"])
pulmon = _recortar(pulmon_full, tuple(d["semilla_completo"]), lado, False)
ent, (a_int, a_ext) = E.caracteristicas_entorno(img, seg, pulmon)
fig, axs = plt.subplots(1, 3, figsize=(13, 4.2))
axs[0].imshow(d["corte_completo_rem"], **VENT); axs[0].imshow(np.ma.masked_where(~pulmon_full, pulmon_full), cmap="cool", alpha=0.35)
axs[0].set_title("Máscara de pulmón (corte completo)")
rgba = np.zeros(img.shape + (4,)); rgba[a_int] = (0, .6, 1, .5); rgba[a_ext] = (1, .6, 0, .5)
axs[1].imshow(img, **VENT); axs[1].imshow(rgba); contorno(axs[1], seg, "red", 1.5)
axs[1].set_title("Anillo 0–3 mm (azul) y 3–6 mm (naranja)")
axs[2].hist(img[a_int], bins=40, alpha=.6, label="anillo interno", color="tab:blue")
axs[2].hist(img[a_ext], bins=40, alpha=.6, label="anillo externo", color="tab:orange")
axs[2].axvline(C.HU_DENSO_ENTORNO, color="k", ls="--"); axs[2].legend(fontsize=8); axs[2].set_xlabel("HU")
axs[2].set_title("HU en los anillos (a la derecha de la línea = estructuras densas)")
sin_ejes(axs[:2]); plt.tight_layout(); plt.show()
print({k: round(v, 3) for k, v in ent.items() if "glcm" not in k})

### Estructuras que convergen (construido sobre la esqueletización)

La materia usa el **esqueleto** para vasos: largo y ramificaciones. Acá: las estructuras densas del entorno (vasos, espículas, tractos; HU > −500 a menos de 6 mm del nódulo) se reducen a su esqueleto, y se cuenta **cuántas llegan hasta el borde del nódulo**. Es el signo de "convergencia": un nódulo maligno tiende a traccionar el tejido y atraer vasos.

In [ ]:
from skimage.morphology import skeletonize
dist = ndi.distance_transform_edt(~seg) * C.PIXEL_MM
densas = (img > C.HU_DENSO_ENTORNO) & (dist > 0) & (dist <= 6) & pulmon
esq = skeletonize(densas)
lab_e = measure.label(esq, connectivity=2)
toca = set(np.unique(lab_e[esq & (dist <= 1.5 * C.PIXEL_MM)])) - {0}
fig, axs = plt.subplots(1, 2, figsize=(9, 4.3))
axs[0].imshow(img, **VENT); contorno(axs[0], seg, "red", 1.2)
axs[0].imshow(np.ma.masked_where(~densas, densas), cmap="autumn", alpha=.5); axs[0].set_title("Estructuras densas a < 6 mm")
rgb = np.zeros(img.shape + (4,)); rgb[esq] = (0, .7, 1, 1)
for t in toca: rgb[lab_e == t] = (1, 1, 0, 1)
axs[1].imshow(img, **VENT); axs[1].imshow(rgb); contorno(axs[1], seg, "red", 1.2)
axs[1].set_title(f"Esqueleto: {len(toca)} estructuras llegan al nódulo (amarillo)")
sin_ejes(axs); plt.tight_layout(); plt.show()

---
# 8. Pre-procesamiento (obligatorio por consigna)

## 8.1 Reconstrucción tomográfica simulada

Un tomógrafo **no saca fotos**: mide cuánto se atenúa un haz de rayos X al atravesar el cuerpo desde muchos ángulos, y una computadora **reconstruye** la imagen. Simulamos ese proceso al revés y al derecho:

1. **HU → μ**: pasamos a coeficiente de atenuación físico, μ = μ_agua·(1 + HU/1000).
2. **Transformada de Radon**: para cada ángulo, suma μ a lo largo de cada línea → el **sinograma** (cada fila es un ángulo; un punto del cuerpo dibuja una sinusoide, de ahí el nombre).
3. **Ruido de fotones (Beer-Lambert + Poisson)**: llegan I = I₀·e^(−p) fotones; contamos fotones, y esa cuenta tiene ruido de Poisson. **Menos dosis = menos fotones (I₀ chico) = más ruido relativo** (el ruido crece como 1/√I₀).
4. **Reconstrucción** (métodos de la materia):
   - **SBP**: "unta" cada proyección sin filtrar → borrosa con **halo** (sobra baja frecuencia). Solo didáctica.
   - **FBP con rampa (Ram-Lak)**: filtra y retroproyecta. Nítida, pero la rampa **amplifica el ruido** → para datos limpios.
   - **FBP con Hann**: rampa con ventana que frena las frecuencias altísimas → **para datos ruidosos** (baja dosis).
   - **SART**: iterativa; corrige una vista entera por vez promediando → tolera **pocas proyecciones y ruido**. La materia dice que suele bastar **una iteración**: con 180 vistas es cierto; con 60, una iteración todavía no llegó (subestima las HU), por eso mostramos 1 y 3.
   - **ART no**: necesita armar la matriz A (un rayo por fila, un píxel por columna): ~92.000 × 262.144 para un corte de 512×512. SART resuelve el mismo A·x = b sin armarla.

In [ ]:
orig = np.maximum(d["corte_completo"], -1000).astype(float)
px = float(d["pixel_mm_original"])
mu = R.hu_a_mu(orig)
p = R.sinograma(mu, px, 180)
rng = np.random.default_rng(0)
p_ruido = R.agregar_ruido(p, 1e4, rng)

fig, axs = plt.subplots(1, 4, figsize=(16, 4))
axs[0].imshow(mu, cmap="gray"); axs[0].set_title("μ (mm⁻¹)")
axs[1].imshow(p, aspect="auto", cmap="gray"); axs[1].set_title("Sinograma (180 ángulos)"); axs[1].set_xlabel("ángulo"); axs[1].set_ylabel("posición del detector")
axs[2].imshow(p_ruido - p, aspect="auto", cmap="RdBu", vmin=-.1, vmax=.1); axs[2].set_title("Ruido agregado con I₀ = 10⁴ (baja dosis)")
cnt = 1e4 * np.exp(-p); axs[3].plot(cnt[:, 0]); axs[3].set_yscale("log"); axs[3].set_title("Fotones que llegan (ángulo 0°)")
axs[3].set_xlabel("posición del detector")
sin_ejes(axs[:1]); plt.tight_layout(); plt.show()

In [ ]:
def zoom_nod(x, mm=64):
    esc = C.PIXEL_MM / px; r, c = (np.asarray(d["semilla_completo"]) * esc).astype(int); h = int(mm / px / 2)
    return x[max(r-h, 0):r+h, max(c-h, 0):c+h]

sbp, _ = R.simular(orig, px, n_angulos=180, metodo="sbp")
fbp, _ = R.simular(orig, px, n_angulos=180, metodo="fbp")
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
axs[0].imshow(orig, **VENT); axs[0].set_title("Original")
axs[1].imshow(sbp, cmap="gray", vmin=np.percentile(sbp, 1), vmax=np.percentile(sbp, 99)); axs[1].set_title("SBP: borrosa con halo")
axs[2].imshow(fbp, **VENT); axs[2].set_title("FBP (rampa): el filtro cancela el halo")
sin_ejes(axs); plt.tight_layout(); plt.show()

condiciones = {
    "FBP rampa, dosis ref": dict(n_angulos=180, I0=1e5),
    "FBP rampa, 1/10 dosis": dict(n_angulos=180, I0=1e4),
    "FBP Hann, 1/10 dosis": dict(n_angulos=180, I0=1e4, filtro="hann"),
    "SART 1 it, 1/10 dosis": dict(n_angulos=180, I0=1e4, metodo="sart"),
    "FBP rampa, 60 ángulos": dict(n_angulos=60, I0=1e5),
    "SART 1 it, 60 ángulos": dict(n_angulos=60, I0=1e5, metodo="sart"),
    "SART 3 it, 60 ángulos": dict(n_angulos=60, I0=1e5, metodo="sart", iteraciones=3),
}
recs = {k: R.simular(orig, px, rng=np.random.default_rng(0), **kw)[0] for k, kw in condiciones.items()}
fov = R.mascara_fov(orig.shape[0])
fig, axs = plt.subplots(2, 4, figsize=(15, 7.6))
axs = axs.ravel(); axs[0].imshow(zoom_nod(orig), **VENT); axs[0].set_title("Original")
for a, (k, v) in zip(axs[1:], recs.items()):
    m = Z.metricas(np.clip(orig, -1000, 400), np.clip(v, -1000, 400), 1400, fov)
    a.imshow(zoom_nod(v), **VENT); a.set_title(f"{k}\nRMSE {m['rmse']:.0f} HU  SSIM {m['ssim']:.2f}")
sin_ejes(axs); plt.tight_layout(); plt.show()

**Qué mirar:** con menos dosis aparece "granulado"; **Hann** lo reduce mucho respecto de la rampa (tabla de la materia: datos ruidosos → Hann); con pocos ángulos aparecen **rayas** (submuestreo angular), que **SART** tolera mejor. SART y Hann quedan más limpios pero más **suaves**: ¿la textura sobrevive? (hipótesis 3, sección 8.4).

**Métricas:** RMSE = error promedio en HU (menos es mejor). SSIM = similitud estructural de 0 a 1 (compara bordes y contraste local). Usamos las dos porque RMSE no distingue entre "ruido" y "borroneo": una imagen suavizada puede tener RMSE bajo y haber perdido los bordes.

## 8.2 Mejoramiento: filtros sobre la imagen de baja dosis

- **Mediana 3×3**: reemplaza cada píxel por la mediana de sus vecinos. Ideal para ruido impulsivo; conserva bordes.
- **Gaussiano σ=1**: promedio pesado de los vecinos. Pasa-bajos: saca ruido y también suaviza bordes.

- **Wiener 5×5**: adaptativo; promedia mucho donde la zona es pareja (ahí la variación es ruido) y casi nada en los bordes (ahí la variación es estructura).

Regla de la materia: el filtro según el **tipo de ruido**. El de la TC es granulado y parejo → a priori, **gaussiano**. Pero RMSE no cuenta toda la historia: `comparaciones/filtros.py` separa *cuánto ruido queda* de *cuánto se borronea el borde del nódulo*, y ahí Wiener muestra su ventaja.

In [ ]:
ruidosa = recs["FBP rampa, 1/10 dosis"]
filtradas = {"sin filtro": ruidosa, **{k: f(ruidosa) for k, f in M.FILTROS.items()}}
fig, axs = plt.subplots(1, 5, figsize=(17, 3.8))
axs[0].imshow(zoom_nod(orig), **VENT); axs[0].set_title("Original")
for a, (k, v) in zip(axs[1:], filtradas.items()):
    m = Z.metricas(np.clip(orig, -1000, 400), np.clip(v, -1000, 400), 1400, fov)
    a.imshow(zoom_nod(v), **VENT); a.set_title(f"{k}\nRMSE {m['rmse']:.0f}  SSIM {m['ssim']:.2f}")
sin_ejes(axs); plt.tight_layout(); plt.show()

## 8.3 Compresión y descompresión

1. La tomografía tiene ~12 bits; JPEG solo acepta **8 bits** (256 niveles). Aplicamos la ventana [−1000, 400] → cada nivel ≈ 5,5 HU. Eso ya es una pérdida chica (≈ 5,5/√12 ≈ 1,6 HU de error).
2. **Huffman** (redundancia de código): códigos cortos para los tonos frecuentes. Rinde si el histograma es desparejo (y en TC lo es: mucho aire y mucho tejido).
3. **RLE** (redundancia de píxeles): "valor + repeticiones". Rinde en el aire de afuera del cuerpo (todo 0), pero **adentro el ruido corta todas las corridas** y el archivo puede **agrandarse**.
4. **PNG** (sin pérdida): LZ77 (diccionario de secuencias, familia de LZW) + Huffman: combina las dos ideas y gana.
5. **JPEG** (con pérdida): bloques 8×8 → DCT → **cuantización** (acá se pierde: altas frecuencias = bordes y texturas finas) → zigzag + RLE + Huffman. A calidad baja se ven los **bloques**.

In [ ]:
img8 = Z.hu_a_8bits(orig); hu8 = Z.de_8bits_a_hu(img8)
_, nb_png = Z.comprimir(img8, "PNG")
tabla_c = pd.DataFrame([dict(formato=f, bytes=b, tasa=img8.size / b) for f, b in
                        [("8 bits sin comprimir", img8.size), ("Huffman", Z.bytes_huffman(img8)),
                         ("RLE", Z.bytes_rle(img8)), ("PNG", nb_png)]])
display(tabla_c.round(2))
fig, axs = plt.subplots(2, 5, figsize=(17, 7))
axs[0, 0].imshow(zoom_nod(orig), **VENT); axs[0, 0].set_title("Original"); axs[1, 0].axis("off")
for k, q in enumerate([95, 75, 50, 25], start=1):
    dec, nb = Z.comprimir(img8, "JPEG", q); hj = Z.de_8bits_a_hu(dec)
    axs[0, k].imshow(zoom_nod(hj), **VENT); axs[0, k].set_title(f"JPEG q{q}: {img8.size/nb:.0f}:1 más chico")
    axs[1, k].imshow(zoom_nod(hj - hu8), cmap="RdBu", vmin=-60, vmax=60); axs[1, k].set_title(f"error q{q} (HU)")
sin_ejes(axs); plt.tight_layout(); plt.show()

## 8.4 ¿La textura sobrevive? (robustez)

La materia avisa: *el filtrado cambia la textura* y *JPEG altera la GLCM*. Lo medimos: tomamos **la misma máscara** del nódulo y calculamos sus medidas sobre cada versión de la imagen. Si una medida cambia poco, es robusta; si cambia mucho, sus valores dependen de cómo se adquirió o guardó la imagen más que del nódulo. (Para varios nódulos: `comparaciones/robustez.py` → `figuras/comparaciones/robustez.png`.)

In [ ]:
from tpi.datos import _remuestrear
def roi(x):
    rem = _remuestrear(np.maximum(x, -1024), px, 1)
    return _recortar(rem, tuple(d["semilla_completo"]), lado, -1000.0)
def medir(x):
    r_ = roi(x); v = r_[seg]
    t = F.textura_glcm(r_, seg)
    return {"hu_media": v.mean(), "hu_desvio": v.std(ddof=1),
            **{k.replace("glcm_", ""): t[k] for k in ["glcm_energia", "glcm_entropia", "glcm_contraste", "glcm_homogeneidad"]}}
versiones = {"original": orig, **recs, "gaussiano s/ 1/10 dosis": filtradas["gaussiano_s1"]}
for q in (75, 25):
    versiones[f"JPEG q{q}"] = Z.de_8bits_a_hu(Z.comprimir(img8, "JPEG", q)[0])
rob = pd.DataFrame({k: medir(v) for k, v in versiones.items()}).T
ref = rob.loc["original"]
cambio = rob.copy()
cambio[["hu_media", "hu_desvio"]] = rob[["hu_media", "hu_desvio"]] - ref[["hu_media", "hu_desvio"]]
for c in ["energia", "entropia", "contraste", "homogeneidad"]:
    cambio[c] = 100 * (rob[c] - ref[c]) / abs(ref[c])
cambio.columns = ["Δ media (HU)", "Δ desvío (HU)", "energía %", "entropía %", "contraste %", "homogeneidad %"]
display(cambio.round(1))

**Cómo leerlo:** el ruido de baja dosis **infla el contraste y la entropía** (inventa "textura" que no existe) y baja la energía y la homogeneidad. Hann, SART y el gaussiano lo compensan en parte, pero cambian la textura en el otro sentido (suavizan). JPEG a calidad alta casi no la toca; a calidad baja sí. **Conclusión práctica:** medir siempre sobre la imagen original y en DICOM, sin filtrar ni comprimir con pérdida, igual para todos los nódulos.

---
# 9. Todos los nódulos juntos

Galería con la segmentación de cada nódulo (rojo = algoritmo, verde = consenso). ✓ = aceptable.

In [ ]:
filas = []
n = len(archivos); cols = 6; filas_fig = int(np.ceil(n / cols))
fig, axs = plt.subplots(filas_fig, cols, figsize=(2.7 * cols, 2.9 * filas_fig))
for a in np.ravel(axs): a.axis("off")
for a, ruta in zip(np.ravel(axs), archivos):
    dd = cargar_corte(ruta); ii = info(ruta)
    sg = S.segmentar_nodulo(dd, umbral=T, radio_cierre_mm=RC)
    vv = V.evaluar(sg, dd["consenso"], dd["radiologos"])
    a.imshow(dd["img"], **VENT); contorno(a, dd["consenso"], "lime"); contorno(a, sg, "red")
    a.set_title(f"{ruta.stem.split('_')[0][-4:]} {str(ii.get('grupo','?'))[:5]} {vv['dice']:.2f} {'✓' if vv['aceptable'] else '✗'}", fontsize=8)
    filas.append(dict(nodulo=ruta.stem, grupo=ii.get("grupo"), diametro_eq_mm=F.forma_y_tamano(sg)["diametro_eq_mm"], **vv))
plt.tight_layout(); plt.show()
tabla = pd.DataFrame(filas)
tabla.round(3)

In [ ]:
if tabla["grupo"].notna().any():
    fig, axs = plt.subplots(1, 3, figsize=(14, 3.6))
    g = ["benigno", "sospechoso"]
    axs[0].boxplot([tabla.loc[tabla.grupo == x, "diametro_eq_mm"] for x in g], tick_labels=g); axs[0].set_title("Diámetro equivalente (mm)")
    axs[1].boxplot([tabla.loc[tabla.grupo == x, "dice"] for x in g], tick_labels=g); axs[1].set_title("Dice vs consenso")
    axs[2].scatter(tabla.concordancia_radiologos, tabla.dice, c=np.where(tabla.grupo == "sospechoso", "tab:red", "tab:blue"))
    lim = [min(tabla.dice.min(), tabla.concordancia_radiologos.min()) - .05, 1]
    axs[2].plot(lim, lim, "k--", lw=1); axs[2].set_xlabel("concordancia entre radiólogos"); axs[2].set_ylabel("Dice algoritmo")
    axs[2].set_title("Arriba de la línea = aceptable (rojo sosp., azul benig.)")
    plt.tight_layout(); plt.show()

**Para leer estos gráficos:**

- **Diámetro:** los sospechosos son mucho más grandes. Cualquier medida que crezca con el tamaño va a "separar" los grupos sin aportar nada nuevo. Por eso el control por tamaño (comparar dentro de rangos de diámetro) es central.
- **Dice:** es esperable que los sospechosos se segmenten peor (bordes irregulares, pegados a vasos o pleura); hay que confirmarlo con todos los nódulos. Si pasa, usar solo los aceptables pierde a los sospechosos más difíciles → por eso el análisis se repite también sobre todos (`--todos`).
- **Dispersión:** cada punto es un nódulo; los que caen arriba de la diagonal se segmentaron al menos tan bien como coinciden los radiólogos entre sí.

---
# 10. Resumen

| Etapa de la consigna | Qué hicimos | Dónde está |
|---|---|---|
| Contextualización | LIDC-IDRI, selección con criterios clínicos, distribución de píxel/espesor/diámetro/malignidad | `01_seleccionar_nodulos.py`, `tpi/datos.py` |
| Análisis | histogramas bimodales, umbral de Otsu por región, ruido granulado y chico → no filtrar antes de medir | `04_caracteristicas.py` (figura 04_analisis), sección 2bis |
| Reconstrucción | Radon + Beer-Lambert + ruido Poisson; SBP, FBP rampa y Hann, SART (1 y 3 it); menos dosis y menos ángulos | `tpi/reconstruccion.py`, `comparaciones/reconstruccion.py` |
| Mejoramiento | mediana, gaussiano y Wiener (varios tamaños) sobre baja dosis: ruido residual vs nitidez del borde | `tpi/mejoramiento.py`, `comparaciones/filtros.py` |
| Compresión | ventana a 8 bits, Huffman, RLE, PNG, JPEG q95–q25 | `tpi/compresion.py`, `comparaciones/compresion.py` |
| Robustez | cuánto cambia la textura del nódulo con cada reconstrucción, filtro y compresión | `comparaciones/robustez.py` |
| Segmentación | crecimiento de regiones desde semilla, umbral ajustado, apertura, radio máx, relleno | `tpi/segmentacion.py` |
| Validación | Dice/Jaccard vs consenso, aceptable si Dice ≥ concordancia entre radiólogos; Otsu como comparación | `tpi/validacion.py` |
| Características | área, diámetro, circularidad, solidez, excentricidad, espículas (lo que borra la apertura), nitidez del borde (Sobel), media, desvío, asimetría, curtosis, calcio y aire, GLCM (16 niveles, d=1, 4 direcciones); entorno: anillos por dilatación + estructuras convergentes (esqueleto) | `tpi/caracteristicas.py`, `tpi/entorno.py` |
| Puntaje | combina las k mejores medidas no repetidas; diámetro solo vs interior vs interior + entorno; validación cruzada por paciente, ROC | `tpi/puntaje.py`, `06_puntaje.py` |
| Resultados | boxplots, Mann-Whitney + AUC + Holm, control por tamaño (estratos y Spearman parcial) (falta: más nódulos) | `tpi/estadistica.py`, `05_estadistica.py` |

**Qué construimos encima de la materia:** espículas como lo que borra la apertura, nitidez del borde con Sobel, convergencia con el esqueleto, forma (circularidad, solidez, excentricidad), asimetría y curtosis, y un puntaje que combina medidas.

**Dónde nos apartamos de la materia, y por qué:** criterio de crecimiento "HU > T" en vez de "parecido a la semilla" (más estable); relleno de huecos en vez de cierre (no tapar lóbulos); rango de cuantización por tipo de región (si no, la textura del nódulo cae en 2–3 niveles); forma (circularidad, solidez) además de área; SSIM además de RMSE; SART con 3 iteraciones cuando hay pocas vistas; ART no implementado (matriz A inviable para 512×512). Detalle en `DECISIONES.md`.